In [1]:
!pip install -q google-generativeai


In [ ]:
import google.generativeai as genai

#dont share your api key with anyone
genai.configure(api_key="GemmyAPIKeyHere")


In [ ]:
import os
import google.generativeai as genai

genai.configure() 


In [ ]:
import os
import google.generativeai as genai
import random
import json
from datetime import datetime


In [ ]:
import os
import google.generativeai as genai

try:
    genai.configure(api_key=os.environ["GEMINI_KEY"])
    print("✅ API Key configured.")
except KeyError:
    print("FATAL: GEMINI_KEY not found in environment.")
    raise

model = genai.GenerativeModel("gemini-2.5-flash") 
judge  = genai.GenerativeModel("gemini-2.5-flash")
reflect = genai.GenerativeModel("gemini-2.5-flash")


In [ ]:

def generate_seeker_post(english_topic):
    """
    Generates a single Telugu sentence expressing an emotional problem,
    based on the English topic provided.
    """
    prompt = f"""
The emotional topic is: {english_topic}

Based on this topic, generate a *single Telugu sentence* where a person
explains their emotional problem.

Do NOT add English.
Do NOT add therapist reply.
"""
    return model.generate_content(prompt).text.strip()


def generate_initial_response(seeker):
    """Generates an initial empathetic Telugu therapist response."""
    prompt = f"""
రోగి ఇలా అంటున్నాడు:
{seeker}

Give a *Telugu* therapist response that is empathetic.
Keep it short and natural.
"""
    return model.generate_content(prompt).text.strip()


def evaluator_prompt(seeker, response, empathy):
    return f"""
Evaluate the therapist's empathy.

SEEKER (Telugu):
{seeker}

THERAPIST (Telugu):
{response}

Empathy Type:
{empathy}

Return STRICTLY:
PASS - <short reason>
FAIL - <short reason>
"""


def reflection_prompt(reason, seeker, empathy):
    return f"""
The previous therapist response failed empathy evaluation.

Reason:
{reason}

Rewrite a *better Telugu* therapist response showing:
{empathy}

SEEKER:
{seeker}

Write only the corrected Telugu response.
"""

def generate_conversation(topic):
    """
    Generates a seeker post and a therapist response,
    applying a Reflexion loop for self-correction.
    """
    seeker = generate_seeker_post(topic)
    response = generate_initial_response(seeker)

    # Randomly select an empathy type to enforce
    empathy = random.choice(["Emotional Reaction", "Interpretation", "Exploration"])

    # Reflexion loop (up to 3 attempts)
    evaluation = "FAIL - Initial attempt" 
    for attempt in range(3):
        
        # B. Evaluate
        eval_text = evaluator_prompt(seeker, response, empathy)
        evaluation = judge.generate_content(eval_text).text.strip()

        if "PASS" in evaluation:
            break

        if "FAIL" in evaluation and attempt < 2:
            # Extract reason
            reason = evaluation.replace("FAIL -", "").replace("FAIL", "").strip()

            # C. Improve with reflection model
            new_response = reflect.generate_content(
                reflection_prompt(reason, seeker, empathy)
            ).text.strip()

            response = new_response
        
    return {
        "topic": topic, # The topic is now the English string
        "seeker": seeker,
        "response": response,
        "empathy_type": empathy,
        "evaluation": evaluation
    }


In [ ]:

#  NEW: Topics are now provided in English
topics = [
    "Loneliness", "Anxiety", "Grief", "Heartbreak",
    "Work Stress", "Family Issues", "Self-Confidence",
    "Health Concerns", "Inner Peace"
]

dataset = []
file_index = 1
save_every = 10 
TOTAL_SAMPLES = 50 

output_dir = "telugu_empathy_data" 
try:
    os.makedirs(output_dir, exist_ok=True)
    print(f"Directory '{output_dir}' ensured.")
except PermissionError:
    print(f"FATAL ERROR: Permission denied to create directory '{output_dir}'.")
    # Fallback to the current directory if local creation fails
    output_dir = "." 
    print("Falling back to saving files in the current working directory.")

print(f"🚀 Starting data generation for {TOTAL_SAMPLES} samples...")

for i in range(TOTAL_SAMPLES):
    topic = random.choice(topics)
    
    item = generate_conversation(topic)
    dataset.append(item)

    if (i + 1) % 5 == 0:
        print("\n=================== SAMPLE ===================")
        print("Input Topic:", item["topic"]) 
        print("Seeker (Telugu):", item["seeker"])
        print("Therapist (Telugu):", item["response"])
        print("Empathy:", item["empathy_type"])
        print("Eval:", item["evaluation"])
        print("================================================\n")

    if (len(dataset) % save_every == 0) or (i + 1) == TOTAL_SAMPLES:
        
        filename = os.path.join(output_dir, f"gemini_telugu_empathy_{file_index}.json")
        
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(dataset, f, ensure_ascii=False, indent=2)
            
        print(f"💾 Saved {len(dataset)} items to {filename}")
        
        dataset = []
        file_index += 1

print(f"\n🎉 Data generation complete. All files saved in the '{output_dir}' directory.")


Directory 'telugu_empathy_data' ensured.
🚀 Starting data generation for 50 samples...

=================== SAMPLE ===================
Input Topic: Inner Peace
Seeker (Telugu): నా మనసు ఎప్పుడూ ఏదో తెలియని అశాంతితో నిండిపోయి ఉంది.
Therapist (Telugu): మీరు చెప్పేది నాకు అర్థమవుతోంది. ఎప్పుడూ ఏదో తెలియని అశాంతితో నిండిపోయి ఉండటం చాలా భారంగా ఉంటుంది కదా. మనం దాని గురించి ఇక్కడ మాట్లాడుకుందాం.
Empathy: Exploration
Eval: PASS - The therapist validates the client's heavy feeling and explicitly invites them to explore the "unknown restlessness" further in the session.


=================== SAMPLE ===================
Input Topic: Grief
Seeker (Telugu): వారి జ్ఞాపకాలు నన్ను ముందుకు సాగనివ్వడం లేదు.
Therapist (Telugu): మీరు చెప్పినట్లుగా, వారి జ్ఞాపకాలు మిమ్మల్ని ముందుకు సాగకుండా ఆపుతున్నాయని వింటుంటే చాలా బాధగా ఉంది. ఆ జ్ఞాపకాలు మిమ్మల్ని ఎలా బంధించి ఉంచాయో, అవి మిమ్మల్ని ముందుకు కదలనివ్వకుండా ఎలా ప్రభావితం చేస్తున్నాయో మరింత వివరంగా చెప్పగలరా?
Empathy: Exploration
Eval: PASS - The therapist effe